In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electronics_retailer_clg.silver;

In [0]:
from pyspark.sql.functions import col, trim, when, expr

# ================================
# 1. READ BRONZE TABLE
# ================================

df = spark.table("electronics_retailer_clg.bronze.sales")


# ================================
# 2. CLEAN COLUMN NAMES
# ================================

df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])


# ================================
# 3. TRIM SPACES
# ================================

for c in df.columns:
    df = df.withColumn(c, trim(col(c)))


# ================================
# 4. SAFE DATE PARSING (FIX YOUR ERROR 🔥)
# ================================

df = df.withColumn(
    "order_date",
    expr("""
        coalesce(
            try_to_date(order_date, 'yyyy-MM-dd'),
            try_to_date(order_date, 'dd-MM-yyyy'),
            try_to_date(order_date, 'MM/dd/yyyy')
        )
    """)
)

df = df.withColumn(
    "delivery_date",
    expr("""
        coalesce(
            try_to_date(delivery_date, 'yyyy-MM-dd'),
            try_to_date(delivery_date, 'dd-MM-yyyy'),
            try_to_date(delivery_date, 'MM/dd/yyyy')
        )
    """)
)


# ================================
# 5. FIX DATA TYPES
# ================================

df = df.withColumn("order_number", col("order_number").cast("int")) \
       .withColumn("customerkey", col("customerkey").cast("int")) \
       .withColumn("storekey", col("storekey").cast("int")) \
       .withColumn("productkey", col("productkey").cast("int")) \
       .withColumn("quantity", col("quantity").cast("int"))


# ================================
# 6. HANDLE NULLS (IMPORTANT)
# ================================

# Remove critical nulls
df = df.dropna(subset=["order_number", "productkey", "quantity", "order_date"])

# customerkey can be null → allowed for some analysis
# storekey = null → online orders


# ================================
# 7. REMOVE INVALID DATA
# ================================

df = df.filter(col("quantity") > 0)

# Fix invalid delivery dates
df = df.withColumn(
    "delivery_date",
    when(col("delivery_date") < col("order_date"), None)
    .otherwise(col("delivery_date"))
)


# ================================
# 8. CLEAN CURRENCY CODE
# ================================

df = df.withColumn("currency_code", trim(col("currency_code")))


# ================================
# 9. REMOVE DUPLICATES
# ================================

df = df.dropDuplicates(["order_number", "productkey"])


# ================================
# 10. KEEP ONLY REQUIRED COLUMNS ✅
# ================================

df = df.select(
    "order_number",
    "order_date",
    "delivery_date",
    "customerkey",
    "storekey",
    "productkey",
    "quantity",
    "currency_code"
)


# ================================
# 11. FINAL CHECK
# ================================

display(df)
df.printSchema()


# ================================
# 12. WRITE TO SILVER
# ================================

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("electronics_retailer_clg.silver.sales")

print("✅ Sales cleaned perfectly for your dataset")